In [12]:
REPO <- "https://raw.githubusercontent.com/luisfeliperuas18-tech/PI3-LUISFELIPERUAS-LORENZOLOUZADA-ARTHURDAVINO/main/"   # o repositorio do grupo
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor06.R")   # funcoes do curso
cfg    <- cfg_aula()                                                         # decisoes do curso (limiar = 2)
cp     <- read.csv(paste0(REPO, "estrutura/codigo/corpus/corpus_baixada_santista.csv"),   # o corpus do grupo
                   stringsAsFactors = FALSE, fileEncoding = "UTF-8-BOM")     #   UTF-8 com BOM
docs   <- setNames(cp$texto, cp$id)                                          # vetor nomeado id -> texto
cfg$acentos <- "translit"      # tira acentos em vez de trocar por espaco
ix <- montar(docs, cfg)        # refaz o indice com a limpeza certa
head(ix$vocab[ix$vocab %in% c("municipio", "fundacao", "cubatao")])   # tem que achar as tres palavras inteiras                                                                  # me mandem a saida

[1] "cubatao"   "fundacao"  "municipio"

In [13]:
rs  <- read.csv("qrels_consolidado_resolvidos.csv", stringsAsFactors = FALSE)       # consulta, documento, grau
qs  <- unique(rs$consulta)                                                          # as consultas julgadas
qrels <- lapply(qs, function(q) setNames(rs$grau[rs$consulta == q], rs$documento[rs$consulta == q]))  # graus por consulta
names(qrels) <- qs                                                                  # q01 ... q05
nec <- read.csv(paste0(REPO, "estrutura/codigo/to%20delete/5-entrega-necessidades.csv"),  # as 5 necessidades
                stringsAsFactors = FALSE, fileEncoding = "UTF-8-BOM")
consultas <- setNames(nec$texto_consulta, nec$consulta)                             # q01 -> texto digitado
R_q  <- sapply(qrels, function(g) sum(g >= cfg$limiar))                             # relevantes por consulta
usar <- names(qrels)[R_q > 0]; R_q                                                  # as que entram na avaliacao

q01 q02 q03 q04 q05 
  2   2   2   1   2

In [14]:
rocchio <- function(q, Dr, Dnr = character(0), ix, a = 1, b = 0.75, g = 0.15) {  # q: vetor; Dr, Dnr: nomes
  cr  <- rowMeans(ix$w[, Dr, drop = FALSE])                                # centroide dos relevantes
  cnr <- if (length(Dnr)) rowMeans(ix$w[, Dnr, drop = FALSE]) else 0       # dos nao relevantes; sem nenhum, 0
  a * q + b * cr - g * cnr                                                 # a formula, termo a termo
}
ranking_vetor <- function(q, ix)                  # q: vetor de pesos no espaco de ix$vocab
  sort(apply(ix$w, 2, function(d) cosseno(q, d)), # um cosseno por coluna
       decreasing = TRUE)                         # do maior ao menor

In [15]:
ix_aula <- montar(docs_aula(), cfg_aula())                                       # o indice do curso: so para o teste
q_aula  <- vetor_consulta(tokenizar("modelo de recuperacao"), ix_aula$vocab, ix_aula$idf_tfidf)  # a consulta do curso
names(ranking_vetor(rocchio(q_aula, c("d2", "d3"), "d4", ix_aula), ix_aula))     # tem que dar d3 d2 d1 d6 d8 d5 d7 d4

[1] "d3" "d2" "d1" "d6" "d8" "d5" "d7" "d4"

In [16]:
q    <- "q01"                                                      # a consulta de trabalho
g    <- qrels[[q]]                                                 # os graus dela
q0   <- vetor_consulta(preparar(consultas[[q]], cfg), ix$vocab, ix$idf_tfidf)  # o vetor, com a limpeza do grupo
base <- ranking_vetor(q0, ix)                                      # o ranking base
Dr   <- intersect(names(g)[g >= cfg$limiar], names(docs))          # os relevantes julgados que estao no corpus
Dnr  <- intersect(names(base)[1:5], names(g)[g == 0])              # grau 0 visto no top 5 do base
Dr; Dnr                                                            # confiram antes de seguir

[1] "san_787b827e51" "san_89a217bd1e"

[1] "sv_758239684f"  "san_b93b880263" "sv_651336a6c4"  "san_31603f8e29"

In [17]:
novo <- ranking_vetor(rocchio(q0, Dr, Dnr, ix), ix)                # a consulta reformulada, buscada de novo
round(rbind(base    = metricas(base, g, cfg$limiar),               # antes...
            rocchio = metricas(novo, g, cfg$limiar)), 3)           # ...e depois, na colecao completa

,P@3,AP,RR,nDCG_bin,nDCG_grad
base,0.333,0.667,1,0.832,0.832
rocchio,0.667,1.000,1,1.000,1.000


In [18]:
residual <- function(rk, g, fora, limiar)                               # metricas so no que NAO foi marcado
  metricas(rk[!names(rk) %in% fora], g[!names(g) %in% fora], limiar)    # tira 'fora' do ranking e do gabarito
fora <- c(Dr, Dnr)                                                       # tudo que o usuario marcou
round(rbind(base    = residual(base, g, fora, cfg$limiar),               # base, sem os marcados
            rocchio = residual(novo, g, fora, cfg$limiar)), 3)           # Rocchio, sem os marcados

,P@3,AP,RR,nDCG_bin,nDCG_grad
base,0,NA,0,NA,NA
rocchio,0,NA,0,NA,NA


In [19]:
Dr_v   <- intersect(names(base)[1:5], Dr)                               # so os relevantes que estavam no top 5
novo_v <- ranking_vetor(rocchio(q0, Dr_v, Dnr, ix), ix)                 # Rocchio com o que foi visto
fora_v <- c(Dr_v, Dnr)                                                   # os marcados, agora
Dr_v                                                                     # quem o usuario viu
round(rbind(base    = residual(base,   g, fora_v, cfg$limiar),           # o resto, antes
            rocchio = residual(novo_v, g, fora_v, cfg$limiar)), 3)       # o resto, depois: achou algo NOVO?

[1] "san_89a217bd1e"

,P@3,AP,RR,nDCG_bin,nDCG_grad
base,0.333,1.000,1.000,1.0,1.0
rocchio,0.333,0.333,0.333,0.5,0.5


In [20]:
prf <- function(q0, k, ix, p = c(a = 1, b = 0.75, g = 0.15)) {   # top k viram Dr
  top <- names(ranking_vetor(q0, ix))[1:k]                        # os k primeiros do ranking base
  ranking_vetor(rocchio(q0, top, ix = ix,                         # Rocchio sem Dnr...
                        a = p[["a"]], b = p[["b"]], g = p[["g"]]), ix)  # ...com os pesos de p; busca de novo
}
ap_q <- function(txt, grau, k, ix, cfg) {                         # AP de uma consulta; k = 0: sem PRF
  q0 <- vetor_consulta(preparar(txt, cfg), ix$vocab, ix$idf_tfidf)  # o vetor da consulta
  rk <- if (k == 0) ranking_vetor(q0, ix) else prf(q0, k, ix)       # base ou PRF
  metricas(rk, grau, cfg$limiar)[["AP"]]                            # so o AP
}

In [21]:
dev   <- c("q01", "q02", "q03")                                     # aqui se escolhe k
teste <- c("q04", "q05")                                            # so no fim, uma vez
tab_dev <- sapply(dev, function(q) {                                # para cada consulta dev...
  a <- function(k) ap_q(consultas[[q]], qrels[[q]], k, ix, cfg)     # o AP desta consulta com um k
  c(base = a(0), k2 = a(2), k3 = a(3), k5 = a(5))                   # ...o AP em cada opcao
})                                                                  # fim do sapply
round(tab_dev, 3)                                                   # uma coluna por consulta
round(rowMeans(tab_dev), 3)                                         # o MAP de cada opcao

,q01,q02,q03
base,0.667,0.417,1.000
k2,0.280,0.225,1.000
k3,0.189,0.700,0.833
k5,0.119,0.583,1.000


base    k2    k3    k5 
0.694 0.502 0.574 0.568

In [22]:
top5 <- names(ranking_cosseno(consultas[["q01"]], ix, cfg))[1:5]   # o top 5 base da q01
qrels[["q01"]][top5]                                               # graus deles (NA = nao julgado)

san_89a217bd1e  sv_758239684f san_b93b880263  sv_651336a6c4 san_31603f8e29 
             2              0              0              0              0

In [23]:
substr(docs[["sv_758239684f"]], 1, 250)   # o 2o do base: grau 0
substr(docs[["san_89a217bd1e"]], 1, 250)  # o 1o do base: grau 2

[1] "Vicente» (https://www12.senado.leg.br/noticias/materias/2006/09/22/brasil-foi-o-primeiro-pai s-americano-a-fazer-eleicoes-em-1532-para-a-camara-de-sao-vicente). Senado Federal. 22 de setembro de 2006. Consultado em 14 de março de 2018 13. FERREIRA, T"

[1] "Atribui-se a fundação de Santos a Brás Cubas, sertanista português que, em 1536, recebeu a mais vasta sesmaria do litoral da Capitania de São Vicente. Em 1543, Brás Cubas instalou às margens da baía a Casa de Misericórdia de Todos os Santos para abri"

In [24]:
k_grupo <- 3                                                        # o melhor k da PRF no dev, fixado antes do teste
tab_teste <- sapply(teste, function(q) {                            # para cada consulta de teste...
  a <- function(k) ap_q(consultas[[q]], qrels[[q]], k, ix, cfg)     # o mesmo atalho
  c(base = a(0), prf = a(k_grupo))                                  # ...base e PRF com o k escolhido
})                                                                  # fim do sapply
round(tab_teste, 3); round(rowMeans(tab_teste), 3)                  # por consulta e o MAP

,q04,q05
base,1,0.333
prf,1,0.750


base   prf 
0.667 0.875

In [25]:
writeLines(c(
  "cfg <- list(",
  "  limpar    = TRUE,                           # limpeza da Aula 03",
  "  acentos   = \"translit\",                     # tira acentos (decidido na Aula 06)",
  "  stopwords = character(0),                   # sem lista de stopwords",
  "  k1 = 1.2, b = 0.75,                         # BM25",
  "  limiar    = 2,                              # grau 2 conta como relevante",
  "  # --- Aula 06: realimentacao ---",
  "  prf_k   = 0,                                # 0 = sem pseudo-realimentacao",
  "  rocchio = c(a = 1, b = 0.75, g = 0.15)      # pesos do Rocchio",
  ")"), "config.R")                              # grava o arquivo
source("config.R")                                # recarrega o cfg
cfg$prf_k; cfg$rocchio                            # confiram com a decisao

[1] 0

a    b    g 
1.00 0.75 0.15

In [26]:
ranking_motor <- function(txt, ix, cfg) {                                  # o ranking do motor, com a decisao
  q0 <- vetor_consulta(preparar(txt, cfg), ix$vocab, ix$idf_tfidf)         # o vetor da consulta
  if (cfg$prf_k == 0) ranking_vetor(q0, ix) else prf(q0, cfg$prf_k, ix, cfg$rocchio)  # sem ou com PRF
}                                                                          # fim da funcao
round(head(ranking_motor(consultas[["q01"]], ix, cfg), 5), 3)              # o top 5 que o motor daria

san_89a217bd1e  sv_758239684f san_b93b880263  sv_651336a6c4 san_31603f8e29 
         0.142          0.112          0.109          0.106          0.094

In [27]:
list.files()

[1] "00_FICHA_PROJETO.md"              "aula06_parteD_consolidado.md"    
[3] "config.R"                         "qrels_consolidado_resolvidos.csv"
[5] "sample_data"

In [28]:
file.rename("Ficha do projeto, PI III, Motor d.txt", "00_FICHA_PROJETO.md")             # renomeia a ficha
file.rename("Consolidado, PI III, Aula 06, Par.txt", "aula06_parteD_consolidado.md")    # renomeia o consolidado
list.files()                                                                           # tem que mostrar os dois .md

Warning message in file.rename("Ficha do projeto, PI III, Motor d.txt", "00_FICHA_PROJETO.md"):
“cannot rename file 'Ficha do projeto, PI III, Motor d.txt' to '00_FICHA_PROJETO.md', reason 'No such file or directory'”


[1] FALSE

Warning message in file.rename("Consolidado, PI III, Aula 06, Par.txt", "aula06_parteD_consolidado.md"):
“cannot rename file 'Consolidado, PI III, Aula 06, Par.txt' to 'aula06_parteD_consolidado.md', reason 'No such file or directory'”


[1] FALSE

[1] "00_FICHA_PROJETO.md"              "aula06_parteD_consolidado.md"    
[3] "config.R"                         "qrels_consolidado_resolvidos.csv"
[5] "sample_data"

In [32]:
list.files()                                       # tem que mostrar os 2 .md e o qrels, soltos
anexar_estado("00_FICHA_PROJETO.md")               # o estado do R vai para a ficha
anexar_estado("aula06_parteD_consolidado.md")      # e para o consolidado do grupo

[1] "00_FICHA_PROJETO.md"              "aula06_parteD_consolidado.md"    
[3] "config.R"                         "qrels_consolidado_resolvidos.csv"
[5] "sample_data"

Estado do R anexado a 00_FICHA_PROJETO.md (65 linhas de objetos).

Estado do R anexado a aula06_parteD_consolidado.md (65 linhas de objetos).

